# Notebook 1: Data Loading & Understanding
**Ethiopian Household Wealth Prediction — CRISP-DM Phase 1 & 2**

Loads all 5 ESS/LSMS survey waves and merges housing, labour, enterprise, and shock modules.
**No consumption aggregates are used as features** — `cons_quint` is the target only.

## 1. Setup

In [4]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import pyreadstat
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from data_loader import build_all_waves, wave_summary, feature_coverage

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.3f}'.format)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 2. Load All Waves
Modules merged per household:
- **cons_agg**: geography + demographics + target (`cons_quint`)
- **sect1**: household head (age, sex, education)
- **sect3**: labour market participation
- **sect7** (W1/W3 only): non-farm enterprise ownership
- **sect9** (W1/W3) / **sect8** (W4/W5): housing quality + asset ownership
- **sect8** (W1/W3) / **sect9** (W4/W5): shock exposure (18–20 shock types)

> W4/W5 have fewer housing columns (no roof/wall/water in sect8). These are NaN for those waves.

In [5]:
df = build_all_waves(waves=[1, 3, 4, 5], include_w2=True, save=True)
print(f'Combined dataset: {df.shape[0]:,} households × {df.shape[1]} columns')

Combined dataset: 20,652 households × 38 columns


## 3. Wave Summary

In [3]:
wave_summary(df)

,wave,period,n_households,pct_missing,mean_quintile
0,1,2011-12,3969,2.300,3.270
1,3,2015-16,4954,6.600,3.330
2,4,2018-19,6770,27.800,3.340
3,5,2021-22,4959,32.800,3.420


## 4. Feature Coverage
Shows which features are available across waves. W1/W3 have richer housing; W4/W5 have fewer housing cols.

In [ ]:
fc = feature_coverage(df)
fc

In [ ]:
# Visual: feature coverage heatmap
fig, ax = plt.subplots(figsize=(6, 10))
colors = ['#d32f2f' if v < 50 else '#f57c00' if v < 90 else '#388e3c'
          for v in fc['pct_non_null']]
ax.barh(fc['feature'], fc['pct_non_null'], color=colors)
ax.axvline(100, color='grey', linestyle='--', alpha=0.5)
ax.set_xlabel('% Non-Null')
ax.set_title('Feature Coverage Across All Waves')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../reports/feature_coverage.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Target Distribution
`cons_quint`: wealth quintile 1 (poorest 20%) → 5 (wealthiest 20%). Mild class imbalance — Q5 overrepresented.

In [ ]:
target_dist = df['cons_quint'].value_counts().sort_index()
target_dist.index = [f'Q{i}' for i in target_dist.index]
target_dist.rename('n_households')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: overall distribution
colors = ['#d32f2f','#f57c00','#fbc02d','#388e3c','#1565c0']
td = df['cons_quint'].value_counts().sort_index()
axes[0].bar([f'Q{i}' for i in td.index], td.values, color=colors, edgecolor='white')
axes[0].set_title('Overall Wealth Quintile Distribution')
axes[0].set_xlabel('Quintile (1=poorest, 5=wealthiest)')
axes[0].set_ylabel('Households')
for bar, v in zip(axes[0].patches, td.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+50,
                f'{v:,}', ha='center', fontsize=9)

# Right: by wave
wave_quint = df.groupby(['wave','cons_quint']).size().unstack(fill_value=0)
wave_quint.plot(kind='bar', ax=axes[1], color=colors, edgecolor='white')
axes[1].set_title('Wealth Quintile by Wave')
axes[1].set_xlabel('Wave')
axes[1].set_ylabel('Households')
axes[1].legend([f'Q{i}' for i in range(1,6)], title='Quintile')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('../reports/target_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Geographic Distribution

In [ ]:
region_stats = (
    df.groupby('region', observed=True)['cons_quint']
    .agg(n_households='count', mean_quintile='mean', std_quintile='std')
    .round(3)
    .sort_values('mean_quintile', ascending=True)
)
region_stats

In [ ]:
SETTLEMENT_LABELS = {0:'Urban', 1:'Rural', 2:'Small town', 3:'Large town'}
settle_dist = df['settlement'].map(SETTLEMENT_LABELS).value_counts()
settle_dist

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Regional mean quintile
rs = region_stats.sort_values('mean_quintile')
bar_colors = plt.cm.RdYlGn(
    (rs['mean_quintile'] - rs['mean_quintile'].min()) /
    (rs['mean_quintile'].max() - rs['mean_quintile'].min())
)
axes[0].barh(rs.index, rs['mean_quintile'], color=bar_colors, edgecolor='white')
axes[0].axvline(3, color='grey', linestyle='--', alpha=0.7, label='Q3 midpoint')
axes[0].set_xlabel('Mean Wealth Quintile')
axes[0].set_title('Mean Wealth by Region')
axes[0].legend()

# Settlement distribution
settle_dist.plot(kind='bar', ax=axes[1],
                 color=['#1976D2','#4CAF50','#FF9800','#9C27B0'],
                 edgecolor='white')
axes[1].set_title('Settlement Type Distribution')
axes[1].tick_params(axis='x', rotation=15)
axes[1].set_ylabel('Households')

plt.tight_layout()
plt.savefig('../reports/geographic_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Missing Value Report

In [ ]:
from data_cleaner import DataCleaner
cleaner = DataCleaner()
cleaner.missing_report(df)

## 8. Save Raw Dataset

In [ ]:
df.to_csv('../data/processed/all_waves_clean.csv', index=False)
print(f'Saved: {df.shape[0]:,} rows × {df.shape[1]} cols')